In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
print("✅ Working directory set to project root:", PROJECT_ROOT)


✅ Working directory set to project root: /home/robert/projects/hackaton_test


In [2]:
import pandas as pd

In [3]:
DATAFOLDER = PROJECT_ROOT / "data"
RAW_FOLDER = DATAFOLDER / "raw"
EXTRACTED_FOLDER = DATAFOLDER / "extracted"
RAW_FOLDER.mkdir(parents=True, exist_ok=True)
EXTRACTED_FOLDER.mkdir(parents=True, exist_ok=True)
print("✅ Data folders set up at:", DATAFOLDER)

✅ Data folders set up at: /home/robert/projects/hackaton_test/data


In [4]:
FILES = {
    "2025_Q1": RAW_FOLDER / "personal_wellbeing_loneliness_q1_2025.xlsx",
    "2025_Q2": RAW_FOLDER / "personal_wellbeing_loneliness_q2_2025.xlsx",
    "2025_Q3": RAW_FOLDER / "personal_wellbeing_loneliness_q3_2025.xlsx",
    "2025_Q4": RAW_FOLDER / "personal_wellbeing_loneliness_q4_2025.xlsx",
}

In [5]:
LONELY_CATS = [
    "Often or always",
    "Some of the time",
    "Occasionally",
    "Hardly ever",
    "Never",
]

In [6]:
CAT_RENAME = {
    "Often or always": "often",
    "Some of the time": "some_time",
    "Occasionally": "occasionally",
    "Hardly ever": "hardly_ever",
    "Never": "never",
}

In [7]:
SKIPROWS = 10  # keep consistent with what you used

In [8]:
def extract_loneliness_quarter(path, quarter):
    df = pd.read_excel(path, sheet_name="Table_11", skiprows=SKIPROWS)

    # Region rows only
    df = df[df["Breakdown category"] == "Region"].copy()

    # Keep only substantive response categories
    df = df[df["Response options"].isin(LONELY_CATS)].copy()

    # Clean + numeric
    df["Breakdown"] = df["Breakdown"].astype(str).str.strip()
    df["Response options"] = df["Response options"].astype(str).str.strip()
    df["Estimate (%)"] = pd.to_numeric(df["Estimate (%)"], errors="coerce")  # handles [low] etc

    # Wide pivot: one row per region
    wide = (
        df.pivot_table(
            index="Breakdown",
            columns="Response options",
            values="Estimate (%)",
            aggfunc="first"
        )
        .reset_index()
    )

    # Rename columns
    wide = wide.rename(columns={"Breakdown": "region", **CAT_RENAME})

    # Ensure all 5 columns exist (just in case)
    for col in CAT_RENAME.values():
        if col not in wide.columns:
            wide[col] = pd.NA

    # Add quarter + headline loneliness metric
    wide["quarter"] = quarter
    wide["lonely_at_least_occasionally_pct"] = (
        wide["often"].fillna(0)
        + wide["some_time"].fillna(0)
        + wide["occasionally"].fillna(0)
    ).round(0).astype(int)
    wide["lonely_often_or_sometimes_pct"] = (
        wide["often"].fillna(0) + wide["some_time"].fillna(0)
    ).round(0).astype(int)

    # Reorder
    wide = wide[["quarter", "region", "often", "some_time", "occasionally", "hardly_ever", "never", "lonely_at_least_occasionally_pct", "lonely_often_or_sometimes_pct"]]

    # Sanity checks
    if wide["region"].nunique() != 9:
        print(f"⚠️ {quarter}: expected 9 regions, got {wide['region'].nunique()}")

    return wide

In [9]:
# Build all quarters
lonely_frames = [extract_loneliness_quarter(path, q) for q, path in FILES.items()]
final_loneliness = pd.concat(lonely_frames, ignore_index=True)

print(final_loneliness.shape)  # should be (36, 8)
print(final_loneliness.head())

# Save
OUT_PATH = EXTRACTED_FOLDER / "regional_loneliness_quarterly.csv"
final_loneliness.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}")

(36, 9)
Response options  quarter           region  often  some_time  occasionally  \
0                 2025_Q1    East Midlands      6         21            25   
1                 2025_Q1  East of England      5         18            24   
2                 2025_Q1           London      7         22            25   
3                 2025_Q1       North East      9         19            19   
4                 2025_Q1       North West      7         19            22   

Response options  hardly_ever  never  lonely_at_least_occasionally_pct  \
0                          26     20                                52   
1                          30     20                                47   
2                          29     13                                54   
3                          28     23                                47   
4                          30     19                                48   

Response options  lonely_often_or_sometimes_pct  
0                           

In [10]:
final_loneliness.groupby("quarter")["region"].nunique()


quarter
2025_Q1    9
2025_Q2    9
2025_Q3    9
2025_Q4    9
Name: region, dtype: int64

In [11]:

final_loneliness.isna().sum()

Response options
quarter                             0
region                              0
often                               0
some_time                           0
occasionally                        0
hardly_ever                         0
never                               0
lonely_at_least_occasionally_pct    0
lonely_often_or_sometimes_pct       0
dtype: int64